In [48]:
import torch
from transformers import AutoModelForCausalLM, pipeline, AutoTokenizer
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


# Initializing Model
model_id = 'Qwen/Qwen2-0.5B-Instruct'

print(f"Loading Model {model_id} ....")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype='auto',
    device_map="auto"
)

# Create Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer = tokenizer,
    max_new_tokens = 128,
    temperature = 0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

def create_qwen_prompt (user_input:str, history:list=None):
    if history is None:
        history=[]

    messages = [{
    "role": "system",
    "content": "You are a helpful AI Assistant"
}]
    messages.extend(history)
    messages.append({"role":"user","content":user_input})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

print("Setup COmplete")

Loading Model Qwen/Qwen2-0.5B-Instruct ....


Loading weights: 100%|██████████| 290/290 [00:01<00:00, 160.06it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Setup COmplete


In [49]:
import threading
import requests
import time
from flask import Flask, request, jsonify
from flask_cors import CORS
import logging

In [50]:
if 'llm' not in globals() or 'tokenizer' not in globals():
    raise EnvironmentError("Error: llm and tokenizer not found")

print("Environment setup check passed")

Environment setup check passed


Helper Function

In [51]:
def create_qwen_response(response: str, prompt: str):
    if prompt in response:
        return response.replace(prompt,"").strip()
    if "<|im_start|>assistant" in response:
        return response.split("<|im_start|>assistant")[-1].strip()
    return response.strip()

In [52]:
#Initializing Flask
app = Flask(__name__)
CORS(app)
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

In [53]:
# Routes
@app.route('/chat',methods=['POST'])
def chat():
    try:
        data = request.json
        formatted_prompt = create_qwen_prompt(data.get('message',''))
        raw_response = llm.invoke(formatted_prompt)
        ai_response = create_qwen_response(raw_response,formatted_prompt)
        return jsonify({
            "status": "success",
            "response": ai_response
        })
    except Exception as e:
        return jsonify({
            "error": str(e)
        }),500

In [54]:
# Strat Server

def run_flask():
    print("Flask server started")
    app.run(port=5000, use_reloader=False)

t= threading.Thread(target=run_flask)
t.daemon=True
t.start()
print("Waiting for server to be ready...")
time.sleep(3)

Flask server started
Waiting for server to be ready...
 * Serving Flask app '__main__'
 * Debug mode: off


In [58]:
print("Test General Chat")
print("-"*40)
user_question = str(input("Enter your question"))
try:
    resp = requests.post("http://127.0.0.1:5000/chat", json={'message':user_question}) #user input here
    if resp.status_code == 200:
        print(f"User: Introduct Yourself")
        print(f"Ai: {resp.json()['response']}")
    else:
        print(f"Status Code: {resp.status_code} Error: {resp.text}")

except Exception as e:
    print(f"Failed: {e}")

time.sleep(1)

Test General Chat
----------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: Introduct Yourself
Ai: AI agents are increasingly being utilized for various tasks, such as speech recognition, natural language processing, and decision-making. They can help automate repetitive tasks, improve customer service, assist with data analysis, and provide insights into complex systems.
